# Truthprint — 실증 데이터 핸드오프 (Google Colab)

이 노트북은 **셀을 위에서부터 순서대로 실행**하면, ACL 실측 실험에 필요한 데이터
( 실제 번역문 + 사람 주석 )를 만들어 **zip으로 내려받는 것**까지 끝냅니다.

- 당신이 실제로 판단하는 부분: **STEP 5(주석 검토)** 와, 원하면 STEP 4의 번역 시스템 선택.
- 나머지는 ▶ 실행 버튼만 누르면 됩니다.

> 상단 메뉴 **런타임 → 모두 실행**을 눌러도 되고, 셀마다 ▶ 를 눌러도 됩니다.
> GPU는 필요 없습니다 (런타임 유형: CPU로 충분).

## STEP 0 — 저장소 클론 + 설치

In [ ]:
import os, sys
if not os.path.isdir('/content/truthprint'):
    !git clone https://github.com/leemgs/truthprint /content/truthprint
%cd /content/truthprint/code
!pip install -q -e ".[dev]"
%cd /content/truthprint
sys.path.insert(0, '/content/truthprint/code')  # import truthprint 보장
print('setup done')

## STEP 1 — 내가 만든 코드가 실제로 도는지 확인 (권장)

In [ ]:
%cd /content/truthprint/code
!truthprint selftest
!truthprint challenge
%cd /content/truthprint

## STEP 2 — 작업 폴더 만들기 (예제 복사)

`01_source_items.jsonl` 과 `split.json` 은 그대로 둡니다(제가 제공). 나머지를 채웁니다.

In [ ]:
import shutil, os
SRC = '/content/truthprint/handoff/samples'
WORK = '/content/my_handoff_data'
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
shutil.copytree(SRC, WORK)
print('working folder:', WORK)
print(sorted(os.listdir(WORK)))

## STEP 3 — 번역할 원문 문장 목록 뽑기

In [ ]:
import json, csv
rows = []
with open(f'{WORK}/01_source_items.jsonl', encoding='utf-8') as fh:
    for line in fh:
        r = json.loads(line)
        for sid, text in r['watermarked_text'].items():
            rows.append((r['doc_id'], sid, text))
with open(f'{WORK}/to_translate.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['doc_id','sent_id','source_text']); w.writerows(rows)
print(f'{len(rows)} sentences -> {WORK}/to_translate.csv')
for _, sid, t in rows:
    print(' ', sid, '|', t)

## STEP 4 — 실제 번역 만들기  ⭐

아래 셀은 **무료 Google 번역**(`deep-translator`, API 키 불필요)으로 실제 번역을 수행해
`02_transformations.jsonl` 을 자동으로 채웁니다. EN→KO, EN→HI, 그리고 round-trip(EN→KO→EN).

- 다른 시스템(DeepL/NLLB/GPT/Claude 등)을 쓰고 싶으면, 이 셀 대신 그 시스템의 출력을
  같은 형식으로 저장하고 `system` 값을 실제 이름으로 바꾸면 됩니다.
- 논문 규모를 키우려면 `01_source_items.jsonl` 문서 수를 늘려서 저에게 요청하세요(제가 생성).

In [ ]:
!python -m pip install -q deep-translator
from deep_translator import GoogleTranslator
import json

SYSTEM = 'deep-translator/GoogleTranslator(web)'  # 실제 사용 시스템 이름 기록

def tr(text, src, tgt):
    return GoogleTranslator(source=src, target=tgt).translate(text)

out = []
for doc_id, sid, text in rows:
    ko = tr(text, 'en', 'ko')
    hi = tr(text, 'en', 'hi')
    rt = tr(ko, 'ko', 'en')  # round-trip EN->KO->EN
    out.append({'transform_id': f'{sid}-ko', 'doc_id': doc_id, 'sent_id': sid,
                'transform_type':'translation','direction':'en->ko','system':SYSTEM,
                'params':{}, 'output_text': ko, 'round_trip': False})
    out.append({'transform_id': f'{sid}-hi', 'doc_id': doc_id, 'sent_id': sid,
                'transform_type':'translation','direction':'en->hi','system':SYSTEM,
                'params':{}, 'output_text': hi, 'round_trip': False})
    out.append({'transform_id': f'{sid}-rt', 'doc_id': doc_id, 'sent_id': sid,
                'transform_type':'roundtrip_translation','direction':'en->ko->en',
                'system':SYSTEM,'params':{}, 'output_text': rt, 'round_trip': True})

with open(f'{WORK}/02_transformations.jsonl','w',encoding='utf-8') as fh:
    for r in out:
        fh.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'wrote {len(out)} transformations')
for r in out[:6]:
    print(' ', r['transform_id'], '|', r['output_text'])

## STEP 5 — 사람 주석 (초안 자동 생성 → 당신이 검토·수정)  ⭐

아래 셀은 **초안** `03_annotations.jsonl` 을 만듭니다. 초안은 '번역이 의미를 보존했다'고
가정하고 원문의 불변량을 그대로 채워 넣습니다. **당신은 번역문을 읽고 반드시 검토**하세요:

1. 번역이 의미(극성·수량·시간방향·양태·귀속·인과)를 바꿨다면 해당 필드를 고치고
   `invariant_preserved` 를 `false` 로 바꾼다.
2. 번역이 태(voice)나 시간구 위치를 정규화해 carrier가 사라졌으면 그 carrier의
   `reliable` 을 `false` 로 바꾼다 (검출에서 erasure로 처리됨).

필드 정의는 리포의 `handoff/schemas/SCHEMA_KO.md` 를 참고하세요.

In [ ]:
import json
# 원문 fact(불변량)를 sent_id로 인덱싱
src_inv = {}
src_carriers = {}
with open(f'{WORK}/01_source_items.jsonl', encoding='utf-8') as fh:
    for line in fh:
        r = json.loads(line)
        for fct in r['facts']:
            sid = fct['sent_id']
            inv = {k: fct[k] for k in ['agent','patient','predicate','quantity',
                    'polarity','time_dir','modality','attribution','causation']}
            src_inv[sid] = inv

draft = []
with open(f'{WORK}/02_transformations.jsonl', encoding='utf-8') as fh:
    for line in fh:
        t = json.loads(line)
        sid = t['sent_id']
        draft.append({
            'transform_id': t['transform_id'],
            'annotator_id': 'A1_DRAFT',
            'invariants_observed': dict(src_inv[sid]),
            'carriers_observed': [
                {'carrier':'voice','value':'active','reliable':True},
                {'carrier':'time_position','value':'front','reliable':True}],
            'invariant_preserved': True,
            'notes': 'AUTO-DRAFT: 번역문을 읽고 검토·수정하세요. 의미 변경 시 필드 수정 + invariant_preserved=false; carrier가 사라졌으면 reliable=false.'
        })
with open(f'{WORK}/03_annotations.jsonl','w',encoding='utf-8') as fh:
    for r in draft:
        fh.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'wrote {len(draft)} DRAFT annotations -> 검토 필요')

### (선택) 주석을 표로 편집하기
Colab에서 `03_annotations.jsonl` 을 표로 편집하고 싶으면 아래 셀로 파일을 열어 직접 고친 뒤
다시 저장할 수 있습니다. (또는 왼쪽 파일 탐색기에서 더블클릭해 편집)

In [ ]:
# 간단 확인용 출력
import json
for line in open(f'{WORK}/03_annotations.jsonl', encoding='utf-8'):
    r = json.loads(line)
    print(r['transform_id'], '| preserved=', r['invariant_preserved'],
          '| polarity=', r['invariants_observed']['polarity'])

## STEP 6 — (선택) 사람 의미동일성 / baseline

`04_human_factuality.csv`, `05_baseline_outputs.jsonl` 은 예제 형식대로 채우면 됩니다.
어려우면 건너뛰어도 검증은 통과합니다(권장 항목).

## STEP 7 — 검증기로 형식 점검 (READY 뜰 때까지)

In [ ]:
!python /content/truthprint/handoff/validate_handoff.py /content/my_handoff_data

## STEP 8 — 결과 zip 내려받기 → 나에게 전달

아래 셀을 실행하면 `my_handoff_data.zip` 이 다운로드됩니다. 이 파일을 저에게 주시거나,
리포 브랜치에 올린 뒤 이 세션에 알려주세요:

```
my_handoff_data 채웠고 validate READY 떴어. 실측 실험 돌려서 논문 표 채워줘.
```

In [ ]:
import shutil
shutil.make_archive('/content/my_handoff_data', 'zip', '/content/my_handoff_data')
print('created /content/my_handoff_data.zip')
try:
    from google.colab import files
    files.download('/content/my_handoff_data.zip')
except Exception as e:
    print('수동 다운로드: 왼쪽 파일 탐색기에서 my_handoff_data.zip 우클릭 → 다운로드', e)